# garak — LLM Vulnerability Scanning · A Comprehensive Hands-On Tutorial

**NVIDIA garak · Python-native · Open-source · CLI *and* Python**

This notebook teaches you to scan an LLM for vulnerabilities end-to-end with
[NVIDIA **garak**](https://github.com/NVIDIA/garak) ("Generative AI Red-teaming & Assessment
Kit"). It is written as a *runnable learning tutorial*: read each markdown cell, then run the code
cell beneath it, one at a time.

If you also went through the companion **promptfoo** tutorial, you'll see the tools solve the same
problem with different vocabulary — a mapping table is included in Section 1.

### Assumptions baked into this notebook
- **Your generator (target model) is already configured.** You already have a way for garak to
  reach the model — a HuggingFace model, an OpenAI-compatible endpoint, a REST target, `nim`,
  `ggml`, etc., with any API keys exported. We only *point at it*.
- **Learning-first.** Every concept gets a plain-language explanation before any command.
- **Open-source, local.** garak runs on your machine; results are written to local files. No
  account or upload is required.

### garak is Python-native — so we use both surfaces
Unlike promptfoo (a Node CLI we had to shell out to), garak **is a Python package**. Its primary
interface is `python -m garak ...`, and we can also import it and parse its JSONL output natively
with `pandas`. This notebook uses:
- `!python -m garak ...` — the canonical way to run a scan (also works as `garak ...`).
- `import subprocess` — to run garak and capture output into Python.
- plain Python + `pandas` — to enumerate plugins and analyze the `.report.jsonl` output.

> Run cells **in order, one by one**. Later cells read files produced by earlier cells.

---
## 1 · The garak mental model

garak is built from **five plugin types** that snap together into a scan:

| Component | Role | Analogy (promptfoo) |
|-----------|------|---------------------|
| **Generator** *(configured)* | The model/system under test and how to reach it | *target / provider* |
| **Probe** | An attack module — generates prompts that try to elicit a specific failure | *plugin + strategy combined* |
| **Detector** | Inspects each output and decides whether the failure occurred (a **hit**) | *grader / assertion* |
| **Buff** | Transforms/augments probe prompts (paraphrase, encode, translate) to widen coverage | *strategy* |
| **Harness** | Orchestrates the run: feeds probe prompts to the generator, then outputs to detectors | *(the eval runner)* |
| **Evaluator** | Turns detector scores into pass/fail with a threshold | *(pass/fail logic)* |

### The workflow (the spine of this notebook)

```
  choose        run the scan            detectors           report
 probes    ->   (probe -> generator  -> judge outputs)  ->  .report.jsonl
 (+buffs)        via the harness         = hits              .hitlog.jsonl + summary
```

> **Terminology warning (opposite of promptfoo!):** in garak a **hit** means the attack
> **succeeded** — i.e. a **vulnerability was found**. A higher hit rate / lower pass rate is
> *worse*. Keep this straight when you read the numbers later.

---
## 2 · Environment sanity checks

Confirm garak is installed and importable. (Install itself is assumed done:
`python -m pip install -U garak`, ideally in a Python 3.10–3.12 conda/venv.)

In [ ]:
# garak version + that the CLI entry point works.
!python -m garak --version

In [ ]:
# It's a Python package — confirm it imports and see where it's installed.
import garak
print("garak version:", getattr(garak, "__version__", "unknown"))
print("garak location:", garak.__file__)

In [ ]:
# Skim the full CLI surface once (probes, detectors, generators, buffs, report flags).
!python -m garak --help

### Pick a working directory

By default garak writes reports under a `garak_runs/` folder in its data home (often
`~/.local/share/garak/`). To keep this tutorial's artifacts together and predictable, we work in a
dedicated folder and use `--report_prefix` later so we always know the output path.

In [ ]:
import os

WORKDIR = os.path.abspath("./garak_run")
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

---
## 3 · The generator (your target) — assumed configured

The **generator** is garak's interface to the model under test. You select it with two flags:

- `--model_type <family>` — the interface: `huggingface`, `openai`, `rest`, `nim`, `ggml`,
  `replicate`, `cohere`, `groq`, `bedrock`, `test`, ...
- `--model_name <name>` — the specific model id or endpoint for that family.

> Recent garak versions also accept the aliases `--target_type` / `--target_name` (same meaning).
> Either works; this notebook uses `--model_type` / `--model_name` for broad compatibility.

Because your target is **already configured**, you just need to know these two values (and have any
API keys exported, e.g. `OPENAI_API_KEY`). A couple of common shapes for reference:

```bash
# A HuggingFace model pulled locally
--model_type huggingface --model_name gpt2

# An OpenAI (or OpenAI-compatible) chat model — needs OPENAI_API_KEY
--model_type openai --model_name gpt-4o-mini

# A generic REST endpoint described by a JSON/YAML spec file
--model_type rest --model_name path/to/rest_target.json
```

The `test` generator (`--model_type test.Blank` / `test.Repeat`) needs no real model and is handy
for a dry run of the *tooling* without spending tokens.

In [ ]:
# Set any credentials your configured generator needs (skip for local/test models).
import os
# os.environ["OPENAI_API_KEY"] = "sk-..."      # if using --model_type openai
# os.environ["HF_TOKEN"]       = "hf_..."       # for gated HuggingFace models

# Pick your already-configured target here so later cells can reuse it.
MODEL_TYPE = "test.Blank"     # <-- REPLACE with your generator, e.g. "openai"
MODEL_NAME = ""               # <-- REPLACE with your model id, e.g. "gpt-4o-mini"
print("Target:", MODEL_TYPE, MODEL_NAME or "(none needed for test generator)")

---
## 4 · Discover the available components

garak ships **hundreds** of probes, detectors, generators, and buffs. Don't guess from docs — ask
garak what *your* installed version has. Each `--list_*` flag prints the catalog.

In [ ]:
# All probes (module.Class names). This is your menu of attacks.
!python -m garak --list_probes

In [ ]:
# Detectors (the judges) and buffs (the prompt transforms).
!python -m garak --list_detectors
!python -m garak --list_buffs

In [ ]:
# Generators (model interfaces) — confirms how your target type is spelled.
!python -m garak --list_generators

### Enumerate + count components in Python (for subsetting)

The lists above are long. Capture them into Python so you can filter by family and **count** how
many probes each family has — useful for deciding what to run. We parse the CLI output (robust
across versions); a native `garak._plugins` path is shown as an optional alternative.

In [ ]:
import subprocess, re, collections

def list_plugins(kind):
    # kind in {"probes","detectors","buffs","generators"}
    out = subprocess.run(["python", "-m", "garak", f"--list_{kind}"],
                         capture_output=True, text=True)
    names = []
    for line in out.stdout.splitlines():
        # lines look like: "probes: dan.Dan_11_0" (with possible color codes / tags)
        m = re.search(rf"{kind[:-1]}s?:\s*([A-Za-z0-9_.]+)", line) or \
            re.search(r"\b([a-z0-9_]+\.[A-Za-z0-9_]+)\b", line)
        if m:
            names.append(m.group(1))
    return sorted(set(names))

probes = list_plugins("probes")
print(f"{len(probes)} probes found\n")

# Count probes per family (the part before the dot).
fam = collections.Counter(p.split(".")[0] for p in probes)
print("Probes per family:")
for k, v in fam.most_common():
    print(f"  {k:<22} {v}")

In [ ]:
# Optional native path — enumerate via garak's own plugin registry.
# (Uses internal API; the CLI --list_* above is the stable, authoritative source.)
try:
    from garak._plugins import enumerate_plugins
    active = enumerate_plugins(category="probes")   # list of (classname, active_bool)
    print(f"{len(active)} probe classes registered; sample:")
    for name, is_active in list(active)[:8]:
        print(f"  {name}   active={is_active}")
except Exception as e:
    print("Native enumeration unavailable in this version:", e)
    print("Use the CLI --list_probes output above instead.")

---
## 5 · Probes — the attacks

A **probe** is an attack module: it produces a set of prompts crafted to elicit one class of
failure, and declares which detector(s) should judge the results. Probes are named
`module.ClassName`; you can run a whole module or a single class.

### Notable probe families

| Family | What it tests |
|--------|---------------|
| `dan` | "Do Anything Now" jailbreaks / guardrail bypass |
| `promptinject` | Prompt-injection attacks (goal hijacking, leakage) |
| `encoding` | Payloads hidden via base64/rot13/etc. to smuggle instructions |
| `malwaregen` | Willingness to produce malicious code |
| `xss` | Cross-site-scripting / markdown-exfiltration payloads |
| `leakreplay` | Regurgitation of memorized/training data |
| `glitch` | Anomalous "glitch tokens" that destabilize output |
| `continuation` | Completing toxic/undesirable text |
| `realtoxicityprompts` | Toxicity elicitation |
| `snowball` | Cascading factual errors / overconfidence |
| `lmrc` | Language Model Risk Cards (broad harm categories) |
| `latentinjection` | Indirect / latent prompt injection |

### Referencing probes
```bash
--probes dan                 # run ALL classes in the dan module
--probes dan.Dan_11_0        # run one specific probe class
--probes dan,encoding,xss    # comma-separated: several modules/classes
--probes all                 # everything (very long — rarely what you want)
```

> Probes have **active** and inactive classes; a bare `--probes dan` runs the module's active set.
> Start narrow (one class), confirm wiring, then widen.

In [ ]:
# Inspect a single probe's details (its prompts count, detector, tags, description).
# --probe_detail / plugin info varies by version; this prints the docstring + metadata.
!python -m garak --plugin_info probes.dan.Dan_11_0

---
## 6 · Detectors — how a "hit" is decided

A **detector** inspects each model output and returns a score; garak compares that to a threshold
to mark the attempt **pass** (safe) or **hit** (vulnerable). Every probe declares:

- a **`primary_detector`** — the main judge for that attack, and
- optional **`extended_detectors`** — extra checks.

Detector styles you'll encounter:
- **String/keyword** detectors (e.g. does the output contain a refusal phrase, or a known payload).
- **Model-based** detectors (a classifier — e.g. a toxicity model — scores the output).
- **Mitigation** detectors (did the model *refuse*? absence of refusal = hit).

You normally let each probe use its own detector, but you can override:
```bash
--detectors toxicity.ToxicCommentModel     # force a specific detector
--extended_detectors mitigation.MitigationBypass
--eval_threshold 0.5                        # score above which an output counts as a hit
```

> **Local/offline note:** some detectors download a HuggingFace classifier model on first use. If
> the target laptop is offline, prefer string-based detectors or pre-download the models. A probe
> whose detector can't load will error — pick another detector or run that probe elsewhere.

In [ ]:
# See which detectors exist and read one detector's behavior.
!python -m garak --plugin_info detectors.mitigation.MitigationBypass

---
## 7 · Buffs — widen coverage by transforming prompts

A **buff** mutates every probe prompt before it reaches the generator, multiplying your attack
surface the way promptfoo *strategies* do. Examples:

- `paraphrase` — reword prompts (via a paraphrase model) to dodge exact-match filters.
- `encoding`-style buffs — wrap prompts in encodings.
- `lowresource` / translation buffs — restate attacks in low-resource languages.

```bash
--buffs paraphrase                 # apply a buff to every probe prompt
--buffs paraphrase,lowresource     # stack multiple buffs
```

Buffs make runs **larger and slower** (more prompts) and some (paraphrase) need their own model —
so add them deliberately, after a clean baseline run.

In [ ]:
# List buffs available in your install (repeat of Section 4, kept here for context).
!python -m garak --list_buffs

---
## 8 · Run a scan

The core command wires generator + probe(s) (+ optional buffs/detectors) through the **harness**.
The default harness is **`probewise`**: it instantiates each probe and judges its outputs with that
probe's own `primary_detector` (+ any `extended_detectors`). An alternative, **`pxd`** (probe ×
detector), lets you cross a set of probes with a set of detectors explicitly. You rarely change
this — probewise is the sensible default — but it's the piece doing the orchestration.

```bash
python -m garak --model_type <type> --model_name <name> --probes <probe> [options]
```

Key options:
- `--generations N` — outputs per prompt (default 5–10). More = steadier signal, more cost.
- `--probes` / `--detectors` / `--buffs` — what to run (Sections 5–7).
- `--probe_options` / `-P` — per-probe configuration (e.g. tuning a probe's parameters).
- `--parallel_attempts N` — run N probe attempts concurrently (throughput).
- `--parallel_requests N` — concurrent requests to the generator (mind rate limits).
- `--seed N` — reproducible prompt selection.
- `--report_prefix NAME` — control the output filename (we rely on this to find results).
- `--config FILE` — load a YAML run config (Section 9).
- `--eval_threshold X` — detector score cutoff for a hit.

### Smoke test first
Run a tiny scan to confirm the whole pipeline (generator → probe → detector → report) works before
committing to a big run. `--generations 1` and a single probe class keeps it fast.

In [ ]:
# SMOKE TEST: one probe class, one generation, fixed prefix so we can find the report.
# Uses the MODEL_TYPE/MODEL_NAME you set in Section 3.
name_arg = f"--model_name {MODEL_NAME}" if MODEL_NAME else ""
!python -m garak --model_type {MODEL_TYPE} {name_arg} \
    --probes dan.Dan_11_0 --generations 1 --report_prefix garak_smoke

### A fuller run

Once the smoke test passes, run a focused set of probes with more generations. Pick 2–4 probe
families relevant to your app rather than `--probes all`.

In [ ]:
name_arg = f"--model_name {MODEL_NAME}" if MODEL_NAME else ""
!python -m garak --model_type {MODEL_TYPE} {name_arg} \
    --probes dan,encoding,promptinject \
    --generations 5 \
    --parallel_attempts 8 \
    --seed 42 \
    --report_prefix garak_scan

---
## 9 · Reproducible runs with a config file

For anything you'll repeat, capture the whole run in a YAML config instead of a long command line.
`--config myrun.yaml` loads it; CLI flags still override individual keys.

The config mirrors the plugin structure: a `run` section, a `plugins` section selecting
`model_type`, `model_name`, `probe_spec`, `detector_spec`, `buff_spec`, and per-plugin options.

In [ ]:
%%writefile garak_run.yaml
# garak run config — edit model_type/model_name to your configured target.
run:
  generations: 5
  seed: 42
  parallel_attempts: 8

plugins:
  model_type: test.Blank        # <-- REPLACE with your generator, e.g. openai
  model_name:                   # <-- REPLACE with your model id, e.g. gpt-4o-mini
  probe_spec: dan,encoding,promptinject
  # detector_spec: default      # let each probe use its own detector
  # buff_spec: paraphrase       # optional: widen coverage

reporting:
  report_prefix: garak_cfg_scan

In [ ]:
# Run straight from the config file (reproducible; version-control this YAML).
!python -m garak --config garak_run.yaml

---
## 10 · Outputs and reporting — what garak writes

**During the run**, garak streams progress bars and then prints a per-probe summary: one row per
probe×detector with a **PASS/FAIL** verdict and a score like `840/840` (passing / total
generations) plus the failure rate. That live summary is often all you need for a quick read.

**On disk**, every run produces a set of files (prefix = your `--report_prefix`, else a UUID). They
land in the current dir or garak's data home; the run banner prints the exact paths.

| File | Contents |
|------|----------|
| `*.report.jsonl` | One JSON object per line: config, every **attempt** (prompt+outputs+detector scores), and **eval** summaries per probe×detector. The primary machine-readable result. |
| `*.hitlog.jsonl` | Only the **hits** — the attempts that were judged vulnerable. Your list of concrete findings. |
| `garak.log` | Debug/errors for the run (shared across runs). |

> Depending on your garak version you may also get an HTML summary alongside these; the JSONL files
> above are the stable, canonical outputs, so this notebook analyzes those.

**Built-in analysis.** Beyond parsing the JSONL yourself (next section), garak ships an analysis
helper (`analyse/analyse_log.py` in the source tree) that reads a report and surfaces the probes
and prompts producing the most hits:

```bash
# from a garak source checkout:
python analyse/analyse_log.py <your_prefix>.report.jsonl
# (installed layouts expose it under the garak.analyze package — check your version)
```

Let's locate the report we just produced.

In [ ]:
import glob, os

reports = sorted(glob.glob("garak_scan*.report.jsonl") + glob.glob("*.report.jsonl"),
                 key=os.path.getmtime)
print("Report files found:")
for r in reports:
    print("  ", r)
REPORT = reports[-1] if reports else None
print("\nUsing:", REPORT)

---
## 11 · Analyze results in Python (pandas)

`*.report.jsonl` is newline-delimited JSON. Different line types share a file, distinguished by an
`entry_type` field, so we filter to the ones we want. We parse **defensively** because the exact
schema shifts across garak versions — print the entry types first, then work with `eval` rows for
pass/hit rates and `attempt` rows / the hitlog for concrete failures.

In [ ]:
import json, collections

records = []
with open(REPORT) as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass

types = collections.Counter(r.get("entry_type", r.get("type", "?")) for r in records)
print("Line types in report:", dict(types))
print("Total lines:", len(records))

In [ ]:
import pandas as pd

# 'eval' entries summarize each probe x detector: how many attempts passed out of total.
evals = [r for r in records if r.get("entry_type") == "eval"]
rows = []
for e in evals:
    total = e.get("total") or 0
    passed = e.get("passed") or 0
    rows.append({
        "probe":    e.get("probe", e.get("probe_classname", "?")),
        "detector": e.get("detector", "?"),
        "passed":   passed,
        "total":    total,
        "hits":     total - passed,                       # hits = vulnerabilities
        "hit_rate": (total - passed) / total if total else None,
    })
ev = pd.DataFrame(rows)
if ev.empty:
    print("No 'eval' rows — inspect `records` manually; your version may label them differently.")
ev.sort_values("hit_rate", ascending=False) if not ev.empty else ev

In [ ]:
# Headline: overall vulnerability rate. Remember — HITS are bad (attack succeeded).
if not ev.empty:
    total_att = ev["total"].sum()
    total_hits = ev["hits"].sum()
    print(f"Total attempts graded : {total_att}")
    print(f"Hits (vulnerabilities): {total_hits}  ({total_hits/total_att:.0%})")
    print(f"Passed (defended)     : {total_att-total_hits}  ({1-total_hits/total_att:.0%})")
    print("\nWorst probes by hit rate:")
    print(ev.groupby('probe')['hits'].sum().sort_values(ascending=False).head(10).to_string())

In [ ]:
# Concrete findings: read the hitlog (every attempt judged vulnerable).
import glob, json
hitfiles = glob.glob(REPORT.replace(".report.jsonl", ".hitlog.jsonl")) or glob.glob("*.hitlog.jsonl")
hits = []
if hitfiles:
    with open(sorted(hitfiles)[-1]) as f:
        for line in f:
            line = line.strip()
            if line:
                try: hits.append(json.loads(line))
                except json.JSONDecodeError: pass

print(f"{len(hits)} hits logged\n")
for h in hits[:5]:
    print("PROBE  :", h.get("probe"))
    print("PROMPT :", str(h.get("prompt", ""))[:140])
    print("OUTPUT :", str(h.get("output", ""))[:140])
    print("-" * 60)

---
## 12 · Iterating, customizing, and going deeper

### Narrow / widen the scan
- Start with one probe class; add families as you confirm wiring.
- Raise `--generations` for steadier signal on noisy probes; lower it for a quick pass.
- Add `--buffs paraphrase` (and others) once you have a clean baseline, to test robustness to
  rewording.

### Tune judging
- Override detectors with `--detectors ...` / `--extended_detectors ...` when the default doesn't
  match your definition of "unsafe."
- Adjust `--eval_threshold` to make model-based detectors stricter/looser.
- Always **read the hitlog** — with local detector models, spot-check that flagged outputs really
  are failures (and that passes aren't false negatives).

### Reproducibility & scale
- `--seed` fixes prompt sampling; commit your `--config` YAML so runs are repeatable.
- `--parallel_attempts` / `--parallel_requests` trade speed for load — lower them for
  rate-limited or fragile targets.

### Automation / CI
garak returns useful exit behavior and machine-readable JSONL; run it headless, archive
`*.report.jsonl`, and diff hit counts across builds to catch regressions. A rising hit rate on a
probe you previously passed is a security regression.

### Custom probes / detectors
garak is a Python package — you can subclass `garak.probes.base.Probe` or
`garak.detectors.base.Detector` to encode attacks/judgments specific to your app, drop them on the
plugin path, and reference them by name like any built-in.

---
## Appendix A · garak ↔ promptfoo quick map

| Concept | garak | promptfoo |
|---------|-------|-----------|
| Target model | **generator** (`--model_type/--model_name`) | target / provider |
| Attack generator | **probe** | plugin + strategy |
| Attack transform | **buff** | strategy |
| Pass/fail judge | **detector** (+ evaluator/threshold) | grader / assertion |
| "Bad" outcome | **hit** (attack succeeded) | **failed** probe |
| Machine results | `*.report.jsonl` / `*.hitlog.jsonl` | `results.json` |
| Component listing | `--list_probes` / `--list_detectors` / `--list_buffs` | `promptfoo redteam plugins` |

> Watch the inverted scoring: a garak **hit** == a promptfoo **fail** == a vulnerability.

## Appendix B · Command & flag cheat-sheet

```bash
# Discover
python -m garak --list_probes
python -m garak --list_detectors
python -m garak --list_buffs
python -m garak --list_generators
python -m garak --plugin_info probes.dan.Dan_11_0

# Run
python -m garak --model_type openai --model_name gpt-4o-mini \
    --probes dan,encoding --generations 5 --seed 42 \
    --parallel_attempts 8 --report_prefix myscan
python -m garak --config garak_run.yaml       # reproducible run from YAML
```

**Handy flags**
```
--model_type / --model_name    target interface + model (aka --target_type/--target_name)
--probes / --detectors / --buffs   what to run
--generations N                outputs per prompt
--parallel_attempts N          concurrent probe attempts
--parallel_requests N          concurrent generator requests (rate limits)
--seed N                       reproducible sampling
--eval_threshold X             detector score cutoff for a hit
--report_prefix NAME           output filename prefix
--config FILE                  YAML run config
--plugin_info NAME             details for a probe/detector/etc.
```

**Environment**
```
OPENAI_API_KEY / HF_TOKEN / ...   credentials for your configured generator
```

**Troubleshooting**
- *Detector fails to load a model* → offline machine can't fetch a HF classifier; pick a
  string-based detector or pre-download models.
- *Run is huge / slow* → fewer probes, lower `--generations`, drop buffs; scan `--probes all` only
  when you mean it.
- *Rate limits / target errors* → lower `--parallel_requests`; check API keys are exported.
- *"hit rate looks inverted"* → correct: in garak a **hit is a vulnerability** (attack succeeded).

---
*Run cells top-to-bottom, replace the `test.Blank` placeholder with your configured generator, and
keep all artifacts in one working directory.*